# MLP Medium 5-Fold Ensemble


In [1]:
from pathlib import Path
import os, random, json, joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

DATA = Path("data")
MODELS = Path("models")
OUT_DIR = MODELS / "medium_5fold"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA / "train.csv"
TEST_PATH = DATA / "test_features-1.csv"
EXPECTED_PATH = Path("expected_output.csv")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
N_SPLITS = 5
HIDDEN = [128, 64]

print("Device:", DEVICE)


Device: cpu


## Cargar hiperparámetros Medium 


In [ ]:
CONFIG_PATH = MODELS / "medium_ensemble" / "config.joblib"

if CONFIG_PATH.exists():
    old_config = joblib.load(CONFIG_PATH)
    HP = old_config["hyperparams"].copy()
    print("Hiperparámetros recuperados del ensemble anterior.")
else:
    HP = {
        "lr": 1e-3,
        "dropout": 0.1,
        "weight_decay": 1e-4,
        "batch_size": 32,
        "max_epochs": 600,
        "patience": 50,
    }
    print("No se encontró config anterior; usando configuración fallback.")

HP.setdefault("batch_size", 32)
HP.setdefault("max_epochs", 600)
HP.setdefault("patience", 50)

print("HP:", HP)


Hiperparámetros recuperados del ensemble anterior.
HP: {'lr': 0.002, 'dropout': 0.2, 'weight_decay': 0.0001, 'batch_size': 32, 'max_epochs': 600, 'patience': 50}


## Cargar train y test originales


In [3]:
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

assert "SalePrice" in train_df.columns
assert "Id" in train_df.columns
assert "Id" in test_df.columns

X = train_df.drop(columns=["SalePrice", "Id"]).copy()
y_usd = train_df["SalePrice"].to_numpy(dtype=np.float32)

test_ids = test_df["Id"].copy()
X_test_raw = test_df.drop(columns=["Id", "SalePrice"], errors="ignore").copy()

print("Train:", X.shape)
print("Test:", X_test_raw.shape)
print("Target:", y_usd.shape)


Train: (1168, 79)
Test: (292, 79)
Target: (1168,)


## Preprocesamiento


In [ ]:
def make_preprocessor(X_train_raw):
    numeric_cols = X_train_raw.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = X_train_raw.select_dtypes(exclude=[np.number]).columns.tolist()

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
        ("scaler", StandardScaler()),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    return ColumnTransformer([
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ], remainder="drop")


class MLP(nn.Module):
    def __init__(self, input_dim, hidden, dropout=0.0):
        super().__init__()
        layers = []
        prev = input_dim

        for h in hidden:
            layers += [nn.Linear(prev, h), nn.ReLU()]
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [ ]:
def train_fold(Xtr, ytr_scaled, Xva, yva_usd, y_mean, y_std, hp, seed):
    set_seed(seed)

    model = MLP(
        input_dim=Xtr.shape[1],
        hidden=HIDDEN,
        dropout=hp["dropout"]
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=hp["lr"],
        weight_decay=hp["weight_decay"]
    )

    loss_fn = nn.MSELoss()

    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(ytr_scaled, dtype=torch.float32)
    Xva_t = torch.tensor(Xva, dtype=torch.float32, device=DEVICE)

    loader = DataLoader(
        TensorDataset(Xtr_t, ytr_t),
        batch_size=hp["batch_size"],
        shuffle=True,
        generator=torch.Generator().manual_seed(seed)
    )

    best_rmse = float("inf")
    best_state = None
    best_epoch = -1
    wait = 0

    for epoch in range(hp["max_epochs"]):
        model.train()

        for xb, yb in loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()

        model.eval()
        with torch.no_grad():
            pred_scaled = model(Xva_t).cpu().numpy()

        pred_usd = pred_scaled * y_std + y_mean
        rmse = float(np.sqrt(np.mean((pred_usd - yva_usd) ** 2)))

        if rmse < best_rmse - 1e-6:
            best_rmse = rmse
            best_epoch = epoch
            wait = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            wait += 1

        if wait >= hp["patience"]:
            break

    model.load_state_dict(best_state)
    model.eval()

    return model, best_rmse, best_epoch


def predict_usd(model, X_processed, y_mean, y_std):
    with torch.no_grad():
        pred_scaled = model(
            torch.tensor(X_processed, dtype=torch.float32, device=DEVICE)
        ).cpu().numpy()

    return pred_scaled * y_std + y_mean


## Crear los 5 folds


In [6]:
# Bins para estratificación
y_bins = pd.qcut(pd.Series(y_usd), q=10, labels=False, duplicates="drop").to_numpy()

skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED
)

fold_splits = list(skf.split(X, y_bins))

for fold, (tr_idx, va_idx) in enumerate(fold_splits, 1):
    print(f"Fold {fold}: train={len(tr_idx)} val={len(va_idx)}")


Fold 1: train=934 val=234
Fold 2: train=934 val=234
Fold 3: train=934 val=234
Fold 4: train=935 val=233
Fold 5: train=935 val=233


## Entrenar 5 modelos y predecir el test


In [ ]:
oof_pred = np.zeros(len(X), dtype=np.float32)
test_fold_predictions = []
fold_rows = []

for fold, (tr_idx, va_idx) in enumerate(fold_splits, 1):
    print(f"\n{'='*60}")
    print(f"FOLD {fold}/{N_SPLITS}")
    print(f"{'='*60}")

    Xtr_raw = X.iloc[tr_idx].copy()
    Xva_raw = X.iloc[va_idx].copy()

    ytr_usd = y_usd[tr_idx]
    yva_usd = y_usd[va_idx]

    preprocessor = make_preprocessor(Xtr_raw)

    Xtr = preprocessor.fit_transform(Xtr_raw).astype(np.float32)
    Xva = preprocessor.transform(Xva_raw).astype(np.float32)
    Xte = preprocessor.transform(X_test_raw).astype(np.float32)

    y_mean = float(ytr_usd.mean())
    y_std = float(ytr_usd.std(ddof=0))
    ytr_scaled = ((ytr_usd - y_mean) / y_std).astype(np.float32)

    fold_seed = SEED + fold * 100

    model, fold_rmse, best_epoch = train_fold(
        Xtr=Xtr,
        ytr_scaled=ytr_scaled,
        Xva=Xva,
        yva_usd=yva_usd,
        y_mean=y_mean,
        y_std=y_std,
        hp=HP,
        seed=fold_seed
    )

    val_pred = predict_usd(model, Xva, y_mean, y_std)
    test_pred = predict_usd(model, Xte, y_mean, y_std)

    oof_pred[va_idx] = val_pred
    test_fold_predictions.append(test_pred)

    fold_rows.append({
        "fold": fold,
        "n_train": len(tr_idx),
        "n_val": len(va_idx),
        "n_features": Xtr.shape[1],
        "RMSE_val": fold_rmse,
        "best_epoch": best_epoch,
        "seed": fold_seed
    })

    joblib.dump(
        {
            "preprocessor": preprocessor,
            "y_mean": y_mean,
            "y_std": y_std,
            "raw_columns": X.columns.tolist(),
        },
        OUT_DIR / f"fold_{fold}_preprocessor.joblib"
    )

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "hidden_layers": HIDDEN,
            "hyperparams": HP,
            "fold": fold,
            "seed": fold_seed,
            "input_dim": Xtr.shape[1],
            "y_mean": y_mean,
            "y_std": y_std,
            "val_rmse_usd": fold_rmse,
            "best_epoch": best_epoch,
        },
        OUT_DIR / f"fold_{fold}_model.pt"
    )

    print(f"Features: {Xtr.shape[1]}")
    print(f"Best epoch: {best_epoch}")
    print(f"RMSE fold: ${fold_rmse:,.2f}")



FOLD 1/5
Features: 280
Best epoch: 95
RMSE fold: $21,601.55

FOLD 2/5
Features: 285
Best epoch: 0
RMSE fold: $33,732.62

FOLD 3/5
Features: 283
Best epoch: 31
RMSE fold: $24,611.21

FOLD 4/5
Features: 283
Best epoch: 5
RMSE fold: $23,643.91

FOLD 5/5
Features: 283
Best epoch: 1
RMSE fold: $54,537.00


## Evaluación 


In [8]:
fold_df = pd.DataFrame(fold_rows)
display(fold_df)

oof_rmse = float(np.sqrt(np.mean((oof_pred - y_usd) ** 2)))

print(f"\nRMSE OOF 5-Fold: ${oof_rmse:,.2f}")
print(f"RMSE promedio folds: ${fold_df['RMSE_val'].mean():,.2f}")
print(f"Std RMSE folds: ${fold_df['RMSE_val'].std():,.2f}")

fold_df.to_csv(OUT_DIR / "fold_results.csv", index=False)
np.save(OUT_DIR / "oof_predictions.npy", oof_pred)


,fold,n_train,n_val,n_features,RMSE_val,best_epoch,seed
0,1,934,234,280,21601.554688,95,142
1,2,934,234,285,33732.625000,0,242
2,3,934,234,283,24611.207031,31,342
3,4,935,233,283,23643.910156,5,442
4,5,935,233,283,54536.996094,1,542



RMSE OOF 5-Fold: $33,877.00
RMSE promedio folds: $31,625.26
Std RMSE folds: $13,626.83


## Ensemble de los 5 folds para test


In [9]:
test_pred_matrix = np.vstack(test_fold_predictions)
pred_final = test_pred_matrix.mean(axis=0)

print("Matriz predicciones test:", test_pred_matrix.shape)
print("Predicción final:", pred_final.shape)
print("Rango:", float(pred_final.min()), "-", float(pred_final.max()))


Matriz predicciones test: (5, 292)
Predicción final: (292,)
Rango: 52187.79296875 - 507606.09375


## Generar predictions.csv


In [10]:
predictions = pd.DataFrame({
    "Id": test_ids.to_numpy(),
    "Prediction": pred_final
})

assert predictions.columns.tolist() == ["Id", "Prediction"]
assert len(predictions) == len(test_df)
assert predictions["Id"].notna().all()
assert predictions["Prediction"].notna().all()

if EXPECTED_PATH.exists():
    expected = pd.read_csv(EXPECTED_PATH)

    print("Columnas expected:", expected.columns.tolist())

    assert expected.columns.tolist() == ["Id", "Prediction"]
    assert len(expected) == len(predictions)
    assert np.array_equal(
        expected["Id"].to_numpy(),
        predictions["Id"].to_numpy()
    ), "Los Id no coinciden con expected_output.csv"

predictions.to_csv("predictions.csv", index=False)

print("\n✓ predictions.csv generado correctamente")
print("Columnas:", predictions.columns.tolist())
print("Filas:", len(predictions))
display(predictions.head())



✓ predictions.csv generado correctamente
Columnas: ['Id', 'Prediction']
Filas: 292


,Id,Prediction
0,893,149378.796875
1,1106,337372.875000
2,414,101554.343750
3,523,158275.296875
4,1037,345544.187500
